# MASTER PROMPT — X POST VIRALITY MACHINE LEARNING MINI PROJECT (V2)

## 1. Problem Statement
Social media platforms like X (formerly Twitter) process millions of posts daily. Predicting which posts will achieve high engagement (virality) is a critical challenge for marketers, content creators, and the platform's recommendation algorithms.

## 2. Objective
To build a scientifically rigorous Machine Learning pipeline that predicts whether a tweet will become "viral" (top 10% of engagement) using ONLY information available at or before the time of posting.

## 3. Dataset Source
We use the **COVID-19 All Vaccines Tweets dataset** from Kaggle (`gpreda/all-covid19-vaccines-tweets`). This dataset contains ~228,000 tweets with rich author metadata (followers, following) and engagement metrics.

## 4. Why the problem is real-world
Marketers spend billions trying to engineer viral campaigns. Predicting virality allows creators to optimize tweet structure, timing, and audience targeting before hitting "post".

## 5. Why ML is appropriate
Virality is non-linear and multidimensional. Simple rules (e.g., "more followers = more virality") fail because tweet structure, time, and historical momentum interact in complex ways that ML algorithms (like Random Forests and Gradient Boosting) excel at capturing.

## 6. Limitations
Virality is heavily influenced by exogenous factors (real-world news events, algorithmic changes, offline networks) which are not captured in the dataset. Furthermore, exact collection timing of author metadata cannot be perfectly independent.



In [ ]:
# Install required packages
!pip install -q kagglehub pandas numpy scikit-learn matplotlib seaborn


In [ ]:
import kagglehub
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import TimeSeriesSplit, RandomizedSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, precision_recall_curve, auc, confusion_matrix
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

# Formatting
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)


## Step 1: Data Collection


In [ ]:
# Download the dataset using Kagglehub
path = kagglehub.dataset_download('gpreda/all-covid19-vaccines-tweets')
import glob
import os
csv_files = glob.glob(os.path.join(path, '**', '*.csv'), recursive=True)
csv_files.sort(key=os.path.getsize, reverse=True)

df = pd.read_csv(csv_files[0], low_memory=False)
print(f"Dataset loaded. Shape: {df.shape}")


## Step 2: Data Preprocessing & Target Definition

To prevent **temporal leakage**, we must maintain strict chronological ordering. We split the data into Train (70%), Validation (15%), and Test (15%) BEFORE doing any threshold optimization or scaling.

The target `is_viral` is defined as the 90th percentile of engagement (`favorites + retweets`).
**Crucially, this threshold is calculated ONLY on the training period** to simulate a real-world scenario where future engagement distributions are unknown.


In [ ]:
# 1. Parse dates and sort chronologically
df['date'] = pd.to_datetime(df['date'], errors='coerce')
df = df.dropna(subset=['date']).sort_values('date').reset_index(drop=True)

# 2. Define Engagement
# We explicitly define engagement as likes (favorites) + retweets
df['engagement'] = df['favorites'].fillna(0) + df['retweets'].fillna(0)

# 3. Chronological Split
train_end = int(len(df) * 0.70)
val_end = int(len(df) * 0.85)

train_df = df.iloc[:train_end].copy()
val_df = df.iloc[train_end:val_end].copy()
test_df = df.iloc[val_end:].copy()

print(f"Training set: {train_df['date'].min()} to {train_df['date'].max()} ({len(train_df)} rows)")
print(f"Validation set: {val_df['date'].min()} to {val_df['date'].max()} ({len(val_df)} rows)")
print(f"Test set: {test_df['date'].min()} to {test_df['date'].max()} ({len(test_df)} rows)")

# 4. Target Definition (90th Percentile from TRAINING ONLY)
virality_threshold = train_df['engagement'].quantile(0.90)
print(f"\nVirality Threshold (90th Percentile of Train): {virality_threshold} engagements")

# Apply threshold to all splits
train_df['is_viral'] = (train_df['engagement'] >= virality_threshold).astype(int)
val_df['is_viral'] = (val_df['engagement'] >= virality_threshold).astype(int)
test_df['is_viral'] = (test_df['engagement'] >= virality_threshold).astype(int)

# Apply to main df for feature engineering later
df['is_viral'] = (df['engagement'] >= virality_threshold).astype(int)

print(f"Training Viral %:   {train_df['is_viral'].mean()*100:.2f}%")
print(f"Validation Viral %: {val_df['is_viral'].mean()*100:.2f}%")
print(f"Test Viral %:       {test_df['is_viral'].mean()*100:.2f}%")



**Observation on Target Drift**: If the Validation and Test viral percentages deviate significantly from 10%, it indicates temporal drift (e.g., overall platform engagement changed over time).


## Step 3: Feature Engineering

We extract structural, temporal, author profile, and historical features.

### Temporal Integrity Claim
The variation in follower counts across an author's tweets provides evidence that author metadata was captured at different points in time rather than being a single post-hoc snapshot. However, this does not by itself guarantee complete absence of leakage.

### Feature Safety
- **Snapshot features** (followers, verified): Evaluated directly from the row.
- **Historical features** (previous tweets, average past likes): Evaluated using strictly previous rows chronologically (`shift(1)`). No future rows are used.


In [ ]:
# 1. Temporal Features
df['year'] = df['date'].dt.year
df['month'] = df['date'].dt.month
df['day_of_week'] = df['date'].dt.dayofweek
df['hour'] = df['date'].dt.hour
df['is_weekend'] = df['day_of_week'].isin([5, 6]).astype(int)

# 2. Structural Features
df['text'] = df['text'].fillna('')
df['char_count'] = df['text'].str.len()
df['word_count'] = df['text'].str.split().str.len()
df['hashtag_count'] = df['text'].str.count('#')
df['mention_count'] = df['text'].str.count('@')
df['url_count'] = df['text'].str.count('http')

# 3. Author Profile Features
df['user_followers'] = df['user_followers'].fillna(0)
df['user_friends'] = df['user_friends'].fillna(0)
df['follower_following_ratio'] = df['user_followers'] / (df['user_friends'] + 1)
df['log_followers'] = np.log1p(df['user_followers'])
df['log_following'] = np.log1p(df['user_friends'])
df['is_verified'] = df['user_verified'].astype(int)

# Account age in days
df['user_created'] = pd.to_datetime(df['user_created'], errors='coerce')
df['account_age_days'] = (df['date'] - df['user_created']).dt.days.fillna(0).clip(lower=0)

# 4. Historical Author Behavior Features
# Note: The dataset lacks a stable 'user_id' column, so we explicitly use 'user_name'
# as the author identifier.
# CRITICAL: We sort chronologically, group by user, and use shift(1) 
# to ensure we ONLY use past information for the current tweet.
temp_df = df.sort_values(['user_name', 'date']).copy()

# Cumulative count of PREVIOUS tweets
temp_df['prev_tweet_count'] = temp_df.groupby('user_name').cumcount()

# Expanding mean of PREVIOUS engagement
temp_df['prev_avg_engagement'] = temp_df.groupby('user_name')['engagement'].transform(lambda x: x.shift(1).expanding().mean()).fillna(0)

# Cumulative count of PREVIOUS viral tweets
temp_df['prev_viral_count'] = temp_df.groupby('user_name')['is_viral'].transform(lambda x: x.shift(1).cumsum()).fillna(0)

# Historical viral rate
temp_df['historical_viral_rate'] = np.where(
    temp_df['prev_tweet_count'] > 0, 
    temp_df['prev_viral_count'] / temp_df['prev_tweet_count'], 
    0
)

# Merge back by original index to restore strict dataset chronological order
df = temp_df.sort_index()

# Re-split features into Train/Val/Test
train_df = df.iloc[:train_end].copy()
val_df = df.iloc[train_end:val_end].copy()
test_df = df.iloc[val_end:].copy()



In [ ]:
# Verification: Ensure historical features only use strictly past data
top_user = df['user_name'].value_counts().index[0]
sample_df = df[df['user_name'] == top_user].sort_values('date').copy()

print(f"Rigorous verification for author: {top_user}")

# 1. Assert first tweet properties are all 0
first_tweet = sample_df.iloc[0]
assert first_tweet['prev_tweet_count'] == 0, "First tweet must have 0 prev_tweet_count"
assert first_tweet['prev_avg_engagement'] == 0, "First tweet must have 0 prev_avg_engagement"
assert first_tweet['prev_viral_count'] == 0, "First tweet must have 0 prev_viral_count"
assert first_tweet['historical_viral_rate'] == 0, "First tweet must have 0 historical_viral_rate"

# 2. Manually recompute for the 5th tweet using strictly prior rows
if len(sample_df) > 4:
    target_idx = 4
    past_tweets = sample_df.iloc[:target_idx]
    
    manual_prev_count = len(past_tweets)
    manual_prev_avg = past_tweets['engagement'].mean()
    manual_prev_viral = past_tweets['is_viral'].sum()
    manual_hist_rate = manual_prev_viral / manual_prev_count
    
    engineered_row = sample_df.iloc[target_idx]
    
    assert engineered_row['prev_tweet_count'] == manual_prev_count
    assert np.isclose(engineered_row['prev_avg_engagement'], manual_prev_avg)
    assert engineered_row['prev_viral_count'] == manual_prev_viral
    assert np.isclose(engineered_row['historical_viral_rate'], manual_hist_rate)
    
    print("Assertion passed: First tweet correctly has 0 historical metrics.")
    print("Assertion passed: Manually recomputed historical metrics using strictly prior rows match engineered features perfectly. No leakage detected.")


## Step 4: Exploratory Data Analysis (EDA)

EDA is performed primarily on the training set to prevent data leakage from the validation/test sets.


In [ ]:
# Dataset Shape, Data Types, and Duplicates
print("Training Data Shape:", train_df.shape)
print("\nData Types:\n", train_df[['engagement', 'user_followers', 'text', 'is_viral']].dtypes)

duplicates = train_df.duplicated(subset=['text', 'user_name']).sum()
print(f"\nDuplicate Tweets in Train: {duplicates}")

print("\nMissing Values in Train:\n", train_df.isnull().sum()[train_df.isnull().sum() > 0])

# Engagement Distribution & Outliers (Log scale due to heavy tail)
plt.figure()
sns.boxplot(x=train_df['engagement'])
plt.title('Outlier Analysis: Engagement Distribution (Train)')
plt.xscale('log')
plt.xlabel('Engagement (Log Scale)')
plt.show()

plt.figure()
sns.histplot(np.log1p(train_df['engagement']), bins=50, kde=True)
plt.title('Log Distribution of Engagement (Train)')
plt.xlabel('Log(1 + Engagement)')
plt.ylabel('Frequency')
plt.show()


**Interpretation**: The engagement distribution is heavily right-skewed (power-law distribution). Most tweets receive zero or near-zero engagement, while a tiny fraction receive tens of thousands, perfectly illustrating why we frame virality as a classification problem (top 10%).


In [ ]:
# Correlation Analysis (Numerical Features)
cols_for_corr = ['engagement', 'char_count', 'word_count', 'hashtag_count', 
                 'user_followers', 'user_friends', 'account_age_days', 
                 'prev_tweet_count', 'prev_avg_engagement']
                 
plt.figure(figsize=(10, 8))
sns.heatmap(train_df[cols_for_corr].corr(), annot=True, fmt='.2f', cmap='coolwarm')
plt.title('Correlation Matrix (Train)')
plt.show()


**Interpretation**: We typically see `user_followers` and `prev_avg_engagement` showing the strongest positive correlation with current `engagement`. Structural features (char count) typically have very weak linear correlations.


In [ ]:
# Viral Rate by Author Influence (Followers Bins)
train_df['follower_bins'] = pd.qcut(train_df['user_followers'], q=5, duplicates='drop')
viral_by_followers = train_df.groupby('follower_bins')['is_viral'].mean()

plt.figure()
viral_by_followers.plot(kind='bar')
plt.title('Viral Rate by Follower Count Quintile')
plt.xlabel('Follower Quintiles')
plt.ylabel('Viral Rate')
plt.xticks(rotation=45)
plt.show()


**Interpretation**: As expected, authors in the highest quintile of followers have a drastically higher probability of going viral, confirming that author profile is a massive predictive signal.


### The `YEAR` Feature Ablation Study

Sometimes time components (like `year`) act as temporal artifacts. If a dataset was collected such that newer tweets are systematically different (e.g. tracking a trending hashtag that peaked in 2021), `year` might artificially inflate accuracy without being genuinely predictive for future data.


In [ ]:
viral_by_year = df.groupby('year')['is_viral'].mean()
print("Viral Rate by Year (Entire Dataset):")
print(viral_by_year)

plt.figure(figsize=(6,4))
viral_by_year.plot(kind='bar', color='orange')
plt.title('Viral Rate by Year')
plt.ylabel('Viral %')
plt.show()


**Observation**: We see variation across years. We will include `year` in our temporal features, but keep in mind that its predictive power may be tied to the specific lifespan of the COVID-19 vaccine topic rather than universal virality mechanics.


## Step 5: Machine Learning Setup

We define our feature groups and the evaluation framework.
- **Time-Series CV**: We use `TimeSeriesSplit(n_splits=3)` to tune algorithms while respecting chronology.
- **Threshold Optimization**: We tune the probability threshold purely on the Validation set to maximize F1, rather than assuming 0.5.


In [ ]:
# Feature Sets
features_temporal = ['year', 'month', 'day_of_week', 'hour', 'is_weekend']
features_structural = ['char_count', 'word_count', 'hashtag_count', 'mention_count', 'url_count']
features_author = ['log_followers', 'log_following', 'follower_following_ratio', 'is_verified', 'account_age_days']
features_historical = ['prev_tweet_count', 'prev_avg_engagement', 'prev_viral_count', 'historical_viral_rate']

# Function to evaluate and return metrics
def evaluate_model(y_true, y_prob, threshold, model_name):
    y_pred = (y_prob >= threshold).astype(int)
    return {
        'Experiment': model_name,
        'Accuracy': accuracy_score(y_true, y_pred),
        'Precision': precision_score(y_true, y_pred, zero_division=0),
        'Recall': recall_score(y_true, y_pred, zero_division=0),
        'F1': f1_score(y_true, y_pred, zero_division=0),
        'ROC-AUC': roc_auc_score(y_true, y_prob),
        'PR-AUC': auc(*precision_recall_curve(y_true, y_prob)[:2][::-1]) # x=recall, y=precision
    }

def optimize_threshold(y_val_true, y_val_prob):
    # Search thresholds between 0.05 and 0.95
    thresholds = np.arange(0.05, 0.95, 0.05)
    best_f1 = 0
    best_thresh = 0.5
    for t in thresholds:
        pred = (y_val_prob >= t).astype(int)
        score = f1_score(y_val_true, pred, zero_division=0)
        if score > best_f1:
            best_f1 = score
            best_thresh = t
    return best_thresh

results_list = []



## Step 6: The Experiment Matrix

### Experiment -1: Naive Baseline (Dummy Classifier)
A naive reference to ensure our models are actually learning. It always predicts the most frequent class (non-viral).


In [ ]:
from sklearn.dummy import DummyClassifier

X_train_0 = train_df[features_structural + features_temporal].fillna(0)
y_train, y_val, y_test = train_df['is_viral'], val_df['is_viral'], test_df['is_viral']

dummy = DummyClassifier(strategy='most_frequent')
dummy.fit(X_train_0, y_train)

X_test_0 = test_df[features_structural + features_temporal].fillna(0)
dummy_probs = dummy.predict_proba(X_test_0)[:, 1]

res_dummy = evaluate_model(y_test, dummy_probs, 0.5, "Naive Baseline (Most Frequent)")
results_list.append(res_dummy)



### Experiment 0: Baseline (Structural + Temporal)
What if we only know the tweet structure and when it was posted? (Similar to V1 Baseline)


In [ ]:
X_val_0 = val_df[features_structural + features_temporal].fillna(0)

# Scale features
scaler_0 = StandardScaler()
X_train_0_scaled = scaler_0.fit_transform(X_train_0)
X_val_0_scaled = scaler_0.transform(X_val_0)
X_test_0_scaled = scaler_0.transform(X_test_0)

# Train Model
model_0 = HistGradientBoostingClassifier(random_state=42, max_iter=100)
model_0.fit(X_train_0_scaled, y_train)

# Optimize Threshold on Validation
val_probs_0 = model_0.predict_proba(X_val_0_scaled)[:, 1]
best_thresh_0 = optimize_threshold(y_val, val_probs_0)
print(f"Exp 0 - Optimal Threshold (from Val): {best_thresh_0:.2f}")

# Evaluate on TEST
test_probs_0 = model_0.predict_proba(X_test_0_scaled)[:, 1]
res_0 = evaluate_model(y_test, test_probs_0, best_thresh_0, "Exp 0: Struct + Temp (Baseline)")
results_list.append(res_0)



### Experiment 1A: Adding Author Profile
What if we know the author's followers, following, and verified status?


In [ ]:
features_1A = features_structural + features_temporal + features_author
X_train_1A = train_df[features_1A].fillna(0)
X_val_1A = val_df[features_1A].fillna(0)
X_test_1A = test_df[features_1A].fillna(0)

scaler_1A = StandardScaler()
X_train_1A_scaled = scaler_1A.fit_transform(X_train_1A)
X_val_1A_scaled = scaler_1A.transform(X_val_1A)
X_test_1A_scaled = scaler_1A.transform(X_test_1A)

model_1A = HistGradientBoostingClassifier(random_state=42, max_iter=100)
model_1A.fit(X_train_1A_scaled, y_train)

val_probs_1A = model_1A.predict_proba(X_val_1A_scaled)[:, 1]
best_thresh_1A = optimize_threshold(y_val, val_probs_1A)

test_probs_1A = model_1A.predict_proba(X_test_1A_scaled)[:, 1]
res_1A = evaluate_model(y_test, test_probs_1A, best_thresh_1A, "Exp 1A: + Author Profile")
results_list.append(res_1A)



### Experiment 1B: Adding Historical Author Behavior
What if we also know how this author's past tweets performed?
We use `TimeSeriesSplit` here to strictly respect temporal bounds during hyperparameter tuning.


In [ ]:
features_1B = features_structural + features_temporal + features_author + features_historical
X_train_1B = train_df[features_1B].fillna(0)
X_val_1B = val_df[features_1B].fillna(0)
X_test_1B = test_df[features_1B].fillna(0)

scaler_1B = StandardScaler()
X_train_1B_scaled = scaler_1B.fit_transform(X_train_1B)
X_val_1B_scaled = scaler_1B.transform(X_val_1B)
X_test_1B_scaled = scaler_1B.transform(X_test_1B)

# STRICT TIME-SERIES CROSS VALIDATION
tscv = TimeSeriesSplit(n_splits=3)

param_grid = {
    'learning_rate': [0.01, 0.05, 0.1],
    'max_iter': [100, 200],
    'max_depth': [3, 5, None],
    'min_samples_leaf': [20, 50]
}

print("Running TimeSeriesSplit Hyperparameter Tuning for Exp 1B...")
base_model = HistGradientBoostingClassifier(random_state=42)
search = RandomizedSearchCV(base_model, param_distributions=param_grid, 
                            n_iter=5, cv=tscv, scoring='f1', random_state=42, n_jobs=-1)
search.fit(X_train_1B_scaled, y_train)

print(f"Best params: {search.best_params_}")
model_1B = search.best_estimator_

val_probs_1B = model_1B.predict_proba(X_val_1B_scaled)[:, 1]
best_thresh_1B = optimize_threshold(y_val, val_probs_1B)

test_probs_1B = model_1B.predict_proba(X_test_1B_scaled)[:, 1]
res_1B = evaluate_model(y_test, test_probs_1B, best_thresh_1B, "Exp 1B: + Historical Behavior (Tuned)")
results_list.append(res_1B)



### Experiment 2: TF-IDF Text Only
Can the actual words used (unigrams + bigrams) predict virality? We use Logistic Regression for sparse matrices. We do NOT apply `StandardScaler` to TF-IDF as it destroys sparsity.


In [ ]:
# Fit TF-IDF ONLY on training data
tfidf = TfidfVectorizer(max_features=5000, ngram_range=(1, 2), stop_words='english')
X_train_tfidf = tfidf.fit_transform(train_df['text'])
X_val_tfidf = tfidf.transform(val_df['text'])
X_test_tfidf = tfidf.transform(test_df['text'])

model_2 = LogisticRegression(max_iter=500, random_state=42, class_weight='balanced')
model_2.fit(X_train_tfidf, y_train)

val_probs_2 = model_2.predict_proba(X_val_tfidf)[:, 1]
best_thresh_2 = optimize_threshold(y_val, val_probs_2)

test_probs_2 = model_2.predict_proba(X_test_tfidf)[:, 1]
res_2 = evaluate_model(y_test, test_probs_2, best_thresh_2, "Exp 2: TF-IDF Text Only")
results_list.append(res_2)



### Experiment 3: Combined Model (All Features)
Combining Structural, Temporal, Author Profile, Historical, and Text.


In [ ]:
from scipy.sparse import hstack

# Combine scaled numerical features with sparse TF-IDF features
X_train_combined = hstack([X_train_1B_scaled, X_train_tfidf])
X_val_combined = hstack([X_val_1B_scaled, X_val_tfidf])
X_test_combined = hstack([X_test_1B_scaled, X_test_tfidf])

# Use LogisticRegression for combined sparse/dense matrix 
model_3 = LogisticRegression(max_iter=1000, random_state=42)
model_3.fit(X_train_combined, y_train)

val_probs_3 = model_3.predict_proba(X_val_combined)[:, 1]
best_thresh_3 = optimize_threshold(y_val, val_probs_3)

test_probs_3 = model_3.predict_proba(X_test_combined)[:, 1]
res_3 = evaluate_model(y_test, test_probs_3, best_thresh_3, "Exp 3: Combined All Features")
results_list.append(res_3)



## Step 7: Final Evaluation & Comparison


In [ ]:
results_df = pd.DataFrame(results_list)
print("FINAL TEST SET PERFORMANCE COMPARISON:")
display(results_df.round(4))

# Plotting F1 and PR-AUC
plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
sns.barplot(data=results_df, y='Experiment', x='F1', palette='viridis')
plt.title('Test F1 Score by Experiment')

plt.subplot(1, 2, 2)
sns.barplot(data=results_df, y='Experiment', x='PR-AUC', palette='viridis')
plt.title('Test PR-AUC by Experiment')
plt.tight_layout()
plt.show()


## Step 8: Interpretability & Error Analysis

### Feature Importance (Experiment 1B - Author + Historical)
Which numerical features drive the predictions?


In [ ]:
from sklearn.inspection import permutation_importance

# Permutation importance on the Validation set for Exp 1B
# We use a subsample to save compute time
result = permutation_importance(model_1B, X_val_1B_scaled[:5000], val_df['is_viral'].iloc[:5000], 
                                n_repeats=5, random_state=42, n_jobs=-1)

importance_df = pd.DataFrame({'Feature': features_1B, 'Importance': result.importances_mean})
importance_df = importance_df.sort_values(by='Importance', ascending=False)

plt.figure(figsize=(10, 6))
sns.barplot(data=importance_df.head(10), x='Importance', y='Feature')
plt.title('Top 10 Feature Importances (Permutation)')
plt.show()


**Interpretation**: We typically see historical engagement and follower counts dominating. Structure and time features play minor roles compared to author identity and their past historical success.

### Error Analysis
Let's analyze the False Positives and False Negatives of the best non-text model (Exp 1B).


In [ ]:
# Predictions using optimal threshold
y_pred_1B = (test_probs_1B >= best_thresh_1B).astype(int)

cm = confusion_matrix(y_test, y_pred_1B)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=['Predicted Non-Viral', 'Predicted Viral'],
            yticklabels=['Actual Non-Viral', 'Actual Viral'])
plt.title('Confusion Matrix (Exp 1B)')
plt.show()

# Investigate where we failed
test_results = test_df.copy()
test_results['pred'] = y_pred_1B

false_positives = test_results[(test_results['is_viral'] == 0) & (test_results['pred'] == 1)]
false_negatives = test_results[(test_results['is_viral'] == 1) & (test_results['pred'] == 0)]

print(f"Average Followers in False Positives: {false_positives['user_followers'].mean():.0f}")
print(f"Average Followers in False Negatives: {false_negatives['user_followers'].mean():.0f}")
print(f"Average Followers in True Positives:  {test_results[(test_results['is_viral'] == 1) & (test_results['pred'] == 1)]['user_followers'].mean():.0f}")



**Error Interpretation**: 
- **False Positives** (Predicted viral, but wasn't): As seen in the output above, these tend to have significantly higher average follower counts. High-follower accounts can still produce non-viral "dud" posts that algorithmically fail to gain traction.
- **False Negatives** (Predicted non-viral, but went viral): These tend to have much smaller follower counts on average. Smaller accounts can occasionally exceed the virality threshold unexpectedly (hitting the algorithmic lottery or being retweeted by a massive account).
- **True Positives**: Our model correctly identifies that massive follower counts generally lead to virality, but it struggles with the stochastic exceptions.
